# 中证800 V62：主线健康监测与真实 Alpha 审计

这个 notebook 不训练新模型，也不改变组合规则。它接在 V61 后面，专门回答三个问题：

1. 当前 V46/V61 主线在更贴近 JoinQuant 路径的口径下是否仍有稳定超额？
2. `Recall@30`、最终组合命中、随机组合分位、滚动 RankIC 等健康指标是否能及时发现模型失效？
3. 健康状态本身有没有预测价值：Green / Yellow / Red 之后的未来 1/3/6 个月表现是否真的不同？

优先级规则：

- 优先使用 V61 的 `jq_like_*` 日度/月底持仓路径结果。
- 如果没有 JQ-like 结果，降级使用 realized close-to-close 结果。
- 如果 realized 也没有，才使用 proxy 结果，并在 `return_source` 中标注。

注意：`Recall@30_realTop10` 和真实 topK 命中是事后指标，只有未来一个月收益出来后才知道。它们可以用于监控和下一期风险动作验证，但不能在当月调仓时直接使用。

In [ ]:
# ==================== 配置 ====================
import os
import ast
import math
import warnings
from pathlib import Path
from typing import Dict, List, Optional, Tuple, Any

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore", category=RuntimeWarning)

V61_OUT_DIR = Path("csi800_ml_v61_jq_like_portfolio_sim_outputs")
OUT_DIR = Path("csi800_ml_v62_mainline_health_alpha_audit_outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# 如果只审计某一个模型，可以手工填 model_id；None 表示自动选择 V61 summary 中 JQ-like 表现最好的模型。
PRIMARY_MODEL_ID = None

BENCHMARK_CODE = "000906.XSHG"
LOOKBACK_MONTHS = 6
FORWARD_HORIZONS = [1, 3, 6]
RANDOM_N = 500
RANDOM_SEED = 42

DATE_COL = "rebalance_date"
STOCK_COL = "stock"
SCORE_COL = "score"
TARGETS_COL = "targets"

print("V61 input dir:", V61_OUT_DIR.resolve())
print("V62 output dir:", OUT_DIR.resolve())

## 1. 读取 V61 输出

V62 不重新训练模型。它读取以下 V61 产物：

- `v61_model_summary.csv`
- `v61_jq_like_monthly.csv`
- `v61_jq_like_daily_equity.csv`
- `v61_monthly_realized.csv`
- `v61_monthly_portfolio_proxy.csv`
- `v61_score_panel.csv`
- `v61_random_monthly.csv`

如果某些文件不存在，notebook 会尽量降级，但会在输出里标记缺失。

In [ ]:
def read_csv_if_exists(path: Path) -> pd.DataFrame:
    if path.exists():
        df = pd.read_csv(path)
        print("loaded", path.name, df.shape)
        return df
    print("missing", path.name)
    return pd.DataFrame()

model_summary_df = read_csv_if_exists(V61_OUT_DIR / "v61_model_summary.csv")
jq_like_monthly_df = read_csv_if_exists(V61_OUT_DIR / "v61_jq_like_monthly.csv")
jq_like_daily_df = read_csv_if_exists(V61_OUT_DIR / "v61_jq_like_daily_equity.csv")
realized_monthly_df = read_csv_if_exists(V61_OUT_DIR / "v61_monthly_realized.csv")
proxy_monthly_df = read_csv_if_exists(V61_OUT_DIR / "v61_monthly_portfolio_proxy.csv")
score_panel_df = read_csv_if_exists(V61_OUT_DIR / "v61_score_panel.csv")
random_monthly_df = read_csv_if_exists(V61_OUT_DIR / "v61_random_monthly.csv")
latest_targets_df = read_csv_if_exists(V61_OUT_DIR / "v61_latest_targets.csv")

for df in [model_summary_df, jq_like_monthly_df, jq_like_daily_df, realized_monthly_df, proxy_monthly_df, score_panel_df, random_monthly_df]:
    if not df.empty and DATE_COL in df.columns:
        df[DATE_COL] = pd.to_datetime(df[DATE_COL])
    if not df.empty and "date" in df.columns:
        df["date"] = pd.to_datetime(df["date"])

In [ ]:
def choose_primary_model(summary_df: pd.DataFrame, requested: Optional[str] = None) -> Optional[str]:
    if requested is not None:
        return str(requested)
    if summary_df.empty or "model_id" not in summary_df.columns:
        return None
    sort_candidates = [
        "jq_like_cum_ret",
        "jq_like_relative_excess_cum_ret",
        "realized_net_raw_cum_ret",
        "net_raw_cum_ret",
    ]
    tmp = summary_df.copy()
    for col in sort_candidates:
        if col in tmp.columns and tmp[col].notnull().any():
            tmp[col] = pd.to_numeric(tmp[col], errors="coerce")
            best = tmp.sort_values(col, ascending=False).iloc[0]
            print("auto primary model by", col, ":", best["model_id"])
            return str(best["model_id"])
    print("auto primary model by first row:", tmp.iloc[0]["model_id"])
    return str(tmp.iloc[0]["model_id"])

PRIMARY_MODEL_ID = choose_primary_model(model_summary_df, PRIMARY_MODEL_ID)
print("PRIMARY_MODEL_ID =", PRIMARY_MODEL_ID)

if PRIMARY_MODEL_ID is None:
    raise ValueError("没有找到 primary model。请先运行 V61 或手工设置 PRIMARY_MODEL_ID。")

## 2. 构造统一月度收益表

统一收益表的目标是形成一张按 `model_id + rebalance_date` 的月度审计表。

收益优先级：

1. `jq_like_period_ret / jq_like_excess_period_ret`
2. `realized_raw_ret / realized_net_excess_ret`
3. `net_raw_ret / net_excess_ret` 或 proxy 口径

In [ ]:
def parse_targets(x: Any) -> List[str]:
    if isinstance(x, list):
        return [str(i) for i in x]
    if pd.isna(x):
        return []
    s = str(x).strip()
    if not s:
        return []
    for parser in (ast.literal_eval,):
        try:
            obj = parser(s)
            if isinstance(obj, (list, tuple, set)):
                return [str(i) for i in obj]
        except Exception:
            pass
    if ";" in s:
        return [i.strip() for i in s.split(";") if i.strip()]
    if "," in s:
        return [i.strip().strip("'\"") for i in s.split(",") if i.strip()]
    return [s]


def first_existing_col(df: pd.DataFrame, cols: List[str]) -> Optional[str]:
    for c in cols:
        if c in df.columns:
            return c
    return None


def normalize_monthly(df: pd.DataFrame, source: str) -> pd.DataFrame:
    if df.empty:
        return pd.DataFrame()
    out = df.copy()
    if DATE_COL not in out.columns:
        return pd.DataFrame()
    out[DATE_COL] = pd.to_datetime(out[DATE_COL])
    raw_col = first_existing_col(out, ["jq_like_period_ret", "realized_raw_ret", "realized_net_raw_ret", "net_raw_ret", "raw_ret", "period_ret"])
    excess_col = first_existing_col(out, ["jq_like_excess_period_ret", "realized_net_excess_ret", "net_excess_ret", "excess_ret"])
    if raw_col is not None:
        out["audit_raw_ret"] = pd.to_numeric(out[raw_col], errors="coerce")
    else:
        out["audit_raw_ret"] = np.nan
    if excess_col is not None:
        out["audit_excess_ret"] = pd.to_numeric(out[excess_col], errors="coerce")
    else:
        out["audit_excess_ret"] = np.nan
    out["return_source"] = source
    keep = [c for c in ["model_id", "feature_variant", "tag", DATE_COL, TARGETS_COL, "audit_raw_ret", "audit_excess_ret", "return_source"] if c in out.columns]
    return out[keep].copy()

# 逐层合并。收益优先使用高口径，但 targets / tag / feature_variant 允许从低口径回填。
parts = []
parts.append(normalize_monthly(proxy_monthly_df, "proxy"))
parts.append(normalize_monthly(realized_monthly_df, "realized"))
parts.append(normalize_monthly(jq_like_monthly_df, "jq_like"))

monthly_rows = pd.concat([p for p in parts if not p.empty], ignore_index=True) if any(not p.empty for p in parts) else pd.DataFrame()
if monthly_rows.empty:
    raise ValueError("未找到任何可用月度收益表。请先运行 V61。")

source_rank = {"proxy": 0, "realized": 1, "jq_like": 2}
monthly_rows["_source_rank"] = monthly_rows["return_source"].map(source_rank).fillna(-1)

# 先取最高质量收益口径。
monthly_all = monthly_rows.sort_values(["model_id", DATE_COL, "_source_rank"]).drop_duplicates(["model_id", DATE_COL], keep="last").copy()

# 再从所有来源回填 targets/tag/feature_variant，避免 jq_like 文件没有组合成分时丢失健康监测输入。
for fill_col in [TARGETS_COL, "tag", "feature_variant"]:
    if fill_col not in monthly_all.columns:
        monthly_all[fill_col] = np.nan
    if fill_col in monthly_rows.columns:
        fill_map = (monthly_rows.dropna(subset=[fill_col])
                    .sort_values(["model_id", DATE_COL, "_source_rank"])
                    .drop_duplicates(["model_id", DATE_COL], keep="last")
                    .set_index(["model_id", DATE_COL])[fill_col])
        # JoinQuant 旧 pandas 版本没有 pd.MultiIndex.from_frame，用 tuple key 兼容回填。
        keys = list(zip(monthly_all["model_id"], monthly_all[DATE_COL]))
        fill_values = pd.Series([fill_map.get(k, np.nan) for k in keys], index=monthly_all.index)
        monthly_all[fill_col] = monthly_all[fill_col].where(monthly_all[fill_col].notna(), fill_values)

monthly_all = monthly_all.drop(columns=["_source_rank"], errors="ignore")
monthly_all["target_list"] = monthly_all[TARGETS_COL].apply(parse_targets) if TARGETS_COL in monthly_all.columns else [[] for _ in range(len(monthly_all))]

primary_monthly = monthly_all[monthly_all["model_id"].astype(str) == str(PRIMARY_MODEL_ID)].copy().sort_values(DATE_COL)
print("monthly_rows", monthly_rows.shape)
print("monthly_all", monthly_all.shape)
print("primary_monthly", primary_monthly.shape)
display(primary_monthly.tail(12))

## 3. 基础表现与滚动超额审计

这里不叫“CAPM alpha”。对月频多头股票组合，当前更准确的名字是：

- 年化收益
- 年化超额
- 滚动窗口年化超额中位数
- 最大回撤
- drop-top-month stress

In [ ]:
def cum_return(rets: pd.Series) -> float:
    r = pd.to_numeric(rets, errors="coerce").dropna()
    if len(r) == 0:
        return np.nan
    return float((1 + r).prod() - 1)


def annual_return(rets: pd.Series, periods_per_year: int = 12) -> float:
    r = pd.to_numeric(rets, errors="coerce").dropna()
    if len(r) == 0:
        return np.nan
    total = (1 + r).prod() - 1
    years = len(r) / periods_per_year
    if years <= 0 or total <= -1:
        return np.nan
    return float((1 + total) ** (1 / years) - 1)


def max_drawdown_from_returns(rets: pd.Series) -> float:
    r = pd.to_numeric(rets, errors="coerce").fillna(0)
    nav = (1 + r).cumprod()
    dd = nav / nav.cummax() - 1
    return float(dd.min()) if len(dd) else np.nan


def sharpe_monthly(rets: pd.Series) -> float:
    r = pd.to_numeric(rets, errors="coerce").dropna()
    if len(r) < 2 or r.std(ddof=1) == 0:
        return np.nan
    return float(r.mean() / r.std(ddof=1) * np.sqrt(12))


def drop_top_stress(rets: pd.Series, n: int = 1) -> float:
    r = pd.to_numeric(rets, errors="coerce").dropna().sort_values(ascending=False)
    if len(r) <= n:
        return np.nan
    return cum_return(r.iloc[n:])


def rolling_annualized_return(rets: pd.Series, window: int = 36) -> pd.Series:
    r = pd.to_numeric(rets, errors="coerce")
    vals = []
    idxs = []
    for i in range(window - 1, len(r)):
        seg = r.iloc[i-window+1:i+1].dropna()
        idxs.append(r.index[i])
        vals.append(annual_return(seg, periods_per_year=12) if len(seg) >= max(12, window // 2) else np.nan)
    return pd.Series(vals, index=idxs)


def summarize_model_performance(gdf: pd.DataFrame) -> Dict[str, Any]:
    raw = gdf["audit_raw_ret"]
    excess = gdf["audit_excess_ret"]
    rolling_excess = rolling_annualized_return(excess.reset_index(drop=True), window=min(36, max(12, len(gdf)//2))) if len(gdf) >= 12 else pd.Series(dtype=float)
    return {
        "model_id": str(gdf["model_id"].iloc[0]) if "model_id" in gdf.columns and len(gdf) else "",
        "months": int(len(gdf)),
        "start": gdf[DATE_COL].min(),
        "end": gdf[DATE_COL].max(),
        "return_source_last": gdf["return_source"].iloc[-1] if "return_source" in gdf.columns and len(gdf) else "",
        "raw_cum_ret": cum_return(raw),
        "raw_ann_ret": annual_return(raw),
        "raw_sharpe": sharpe_monthly(raw),
        "raw_max_drawdown": max_drawdown_from_returns(raw),
        "excess_cum_ret": cum_return(excess),
        "excess_ann_ret": annual_return(excess),
        "excess_sharpe": sharpe_monthly(excess),
        "excess_max_drawdown": max_drawdown_from_returns(excess),
        "drop_top1_excess_cum_ret": drop_top_stress(excess, 1),
        "drop_top3_excess_cum_ret": drop_top_stress(excess, 3),
        "rolling_excess_ann_median": float(rolling_excess.median()) if len(rolling_excess.dropna()) else np.nan,
        "rolling_excess_ann_min": float(rolling_excess.min()) if len(rolling_excess.dropna()) else np.nan,
    }

model_alpha_audit_df = pd.DataFrame([summarize_model_performance(g) for _, g in monthly_all.sort_values(DATE_COL).groupby("model_id")])
model_alpha_audit_df = model_alpha_audit_df.sort_values(["excess_cum_ret", "raw_cum_ret"], ascending=False)
display(model_alpha_audit_df.head(20))

## 4. 计算健康指标

核心健康指标：

- `recall30_real_top10`：模型 top30 候选池包含多少真实未来 top10。
- `selected_hit_real_top20`：最终组合里有多少只进入真实未来 top20。
- `random_percentile`：最终组合当月收益在同约束随机组合中的分位。
- `rank_ic`：当月分数和未来收益的 Spearman 相关。
- `exposure drift`：目标组合的市值、行业、板块、缺失率等暴露漂移。

In [ ]:
def infer_future_return_col(df: pd.DataFrame) -> Optional[str]:
    candidates = [
        "alpha_1m", "realized_raw_ret", "raw_return_1m", "stock_return_1m", "future_return_1m",
        "return_1m", "label", "y",
    ]
    for c in candidates:
        if c in df.columns and pd.to_numeric(df[c], errors="coerce").notnull().any():
            return c
    numeric_cols = []
    for c in df.columns:
        lc = c.lower()
        if any(k in lc for k in ["alpha", "return", "ret_1m", "future"]):
            if pd.to_numeric(df[c], errors="coerce").notnull().any():
                numeric_cols.append(c)
    return numeric_cols[0] if numeric_cols else None


def spearman_corr(x: pd.Series, y: pd.Series) -> float:
    tmp = pd.DataFrame({"x": x, "y": y}).dropna()
    if len(tmp) < 5 or tmp["x"].nunique() < 2 or tmp["y"].nunique() < 2:
        return np.nan
    return float(tmp["x"].rank().corr(tmp["y"].rank()))

future_col = infer_future_return_col(score_panel_df)
print("future return col inferred from score_panel:", future_col)

if score_panel_df.empty or future_col is None:
    print("score panel 不足，topK recall / rank_ic 将不可用。")
else:
    score_panel_df[DATE_COL] = pd.to_datetime(score_panel_df[DATE_COL])
    score_panel_df[SCORE_COL] = pd.to_numeric(score_panel_df[SCORE_COL], errors="coerce") if SCORE_COL in score_panel_df.columns else np.nan
    score_panel_df[future_col] = pd.to_numeric(score_panel_df[future_col], errors="coerce")

In [ ]:
def board_from_stock_compat(stock: str) -> str:
    s = str(stock)
    if s.startswith("688"):
        return "STAR"
    if s.startswith("30"):
        return "ChiNext"
    if s.startswith(("8", "4", "43", "87", "92")):
        return "BSE"
    if s.startswith("6"):
        return "SH_main"
    if s.startswith(("0", "2")):
        return "SZ_main"
    return "unknown"


def sample_same_board_mix(sdf: pd.DataFrame, targets: List[str], n_iter: int = 500, seed: int = 42) -> np.ndarray:
    """Fallback random baseline: preserve target board mix when possible, otherwise sample same count."""
    rng = np.random.RandomState(seed)
    if future_col is None or sdf.empty or not targets:
        return np.array([])
    tmp = sdf[[STOCK_COL, future_col]].copy()
    tmp[STOCK_COL] = tmp[STOCK_COL].astype(str)
    tmp["board"] = tmp[STOCK_COL].apply(board_from_stock_compat)
    target_boards = pd.Series([board_from_stock_compat(x) for x in targets]).value_counts().to_dict()
    vals = []
    all_idx = np.arange(len(tmp))
    for _ in range(n_iter):
        picked = []
        ok = True
        for board, cnt in target_boards.items():
            pool = tmp.index[tmp["board"] == board].values
            if len(pool) < cnt:
                ok = False
                break
            picked.extend(rng.choice(pool, size=cnt, replace=False).tolist())
        if not ok or len(picked) < len(targets):
            if len(all_idx) < len(targets):
                continue
            picked = rng.choice(all_idx, size=len(targets), replace=False).tolist()
        vals.append(float(pd.to_numeric(tmp.loc[picked, future_col], errors="coerce").mean()))
    return np.asarray(vals, dtype=float)


def build_health_metrics(monthly_df: pd.DataFrame, score_df: pd.DataFrame, random_df: pd.DataFrame, model_id: str) -> pd.DataFrame:
    m = monthly_df[monthly_df["model_id"].astype(str) == str(model_id)].copy().sort_values(DATE_COL)
    rows = []
    rnd = random_df.copy()
    if not rnd.empty and DATE_COL in rnd.columns:
        rnd[DATE_COL] = pd.to_datetime(rnd[DATE_COL])
    rnd_col = first_existing_col(rnd, [
        "random_percentile", "model_random_percentile", "percentile",
        "random_raw_percentile", "random_alpha_percentile",
    ])
    for _, row in m.iterrows():
        dt = row[DATE_COL]
        targets = [str(x) for x in row.get("target_list", [])]
        rec = {
            "model_id": model_id,
            DATE_COL: dt,
            "return_source": row.get("return_source", ""),
            "audit_raw_ret": row.get("audit_raw_ret", np.nan),
            "audit_excess_ret": row.get("audit_excess_ret", np.nan),
            "target_count": len(targets),
            "random_percentile": np.nan,
            "random_percentile_source": "missing",
        }
        if not rnd.empty and rnd_col is not None:
            rmask = (rnd["model_id"].astype(str) == str(model_id)) & (rnd[DATE_COL] == dt) if "model_id" in rnd.columns else (rnd[DATE_COL] == dt)
            rsub = rnd.loc[rmask]
            if len(rsub):
                rec["random_percentile"] = float(pd.to_numeric(rsub[rnd_col], errors="coerce").iloc[0])
                rec["random_percentile_source"] = rnd_col
        if not score_df.empty and future_col is not None and all(c in score_df.columns for c in ["model_id", DATE_COL, STOCK_COL]):
            sdf = score_df[(score_df["model_id"].astype(str) == str(model_id)) & (score_df[DATE_COL] == dt)].copy()
            sdf = sdf.dropna(subset=[future_col])
            if len(sdf) > 0:
                if SCORE_COL in sdf.columns:
                    sdf = sdf.sort_values(SCORE_COL, ascending=False)
                else:
                    sdf = sdf.reset_index(drop=True)
                real_sorted = sdf.sort_values(future_col, ascending=False)
                top10_real = set(real_sorted.head(10)[STOCK_COL].astype(str))
                top20_real = set(real_sorted.head(20)[STOCK_COL].astype(str))
                top30_pred = set(sdf.head(30)[STOCK_COL].astype(str))
                selected = set(targets)
                rec["eligible_count"] = int(len(sdf))
                rec["rank_ic"] = spearman_corr(sdf[SCORE_COL], sdf[future_col]) if SCORE_COL in sdf.columns else np.nan
                rec["recall30_real_top10"] = len(top30_pred & top10_real) / max(len(top10_real), 1)
                rec["recall30_real_top20"] = len(top30_pred & top20_real) / max(len(top20_real), 1)
                rec["selected_hit_real_top10"] = len(selected & top10_real)
                rec["selected_hit_real_top20"] = len(selected & top20_real)
                rec["selected_hit_rate_real_top20"] = len(selected & top20_real) / max(len(selected), 1) if selected else np.nan
                if pd.isna(rec["random_percentile"]) and targets:
                    selected_vals = pd.to_numeric(sdf[sdf[STOCK_COL].astype(str).isin(targets)][future_col], errors="coerce").dropna()
                    random_vals = sample_same_board_mix(sdf, targets, n_iter=RANDOM_N, seed=RANDOM_SEED + int(pd.Timestamp(dt).strftime("%Y%m%d")) % 100000)
                    if len(selected_vals) and len(random_vals):
                        selected_mean = float(selected_vals.mean())
                        rec["random_percentile"] = float((random_vals <= selected_mean).mean())
                        rec["random_percentile_source"] = "fallback_same_board_mix_{}".format(future_col)
            else:
                rec.update({"eligible_count": 0, "rank_ic": np.nan, "recall30_real_top10": np.nan, "recall30_real_top20": np.nan, "selected_hit_real_top10": np.nan, "selected_hit_real_top20": np.nan, "selected_hit_rate_real_top20": np.nan})
        rows.append(rec)
    return pd.DataFrame(rows).sort_values(DATE_COL)

health_monthly_df = build_health_metrics(monthly_all, score_panel_df, random_monthly_df, PRIMARY_MODEL_ID)
display(health_monthly_df.tail(12))

In [ ]:
def board_from_stock(stock: str) -> str:
    s = str(stock)
    if s.startswith("688"):
        return "STAR"
    if s.startswith("30"):
        return "ChiNext"
    if s.startswith(("8", "4", "43", "87", "92")):
        return "BSE"
    if s.startswith("6"):
        return "SH_main"
    if s.startswith(("0", "2")):
        return "SZ_main"
    return "unknown"


def build_exposure_drift(monthly_df: pd.DataFrame, score_df: pd.DataFrame, model_id: str) -> pd.DataFrame:
    if monthly_df.empty:
        return pd.DataFrame()
    rows = []
    m = monthly_df[monthly_df["model_id"].astype(str) == str(model_id)].copy().sort_values(DATE_COL)
    industry_col = first_existing_col(score_df, ["industry", "sw_l1", "industry_name", "jq_l1_industry", "申万一级行业"]) if not score_df.empty else None
    cap_col = first_existing_col(score_df, ["market_cap", "circulating_market_cap", "valuation.market_cap", "log_cap", "size"]) if not score_df.empty else None
    for _, row in m.iterrows():
        dt = row[DATE_COL]
        targets = [str(x) for x in row.get("target_list", [])]
        rec = {"model_id": model_id, DATE_COL: dt, "target_count": len(targets)}
        if targets:
            boards = [board_from_stock(s) for s in targets]
            rec["main_board_ratio"] = sum(b in ("SH_main", "SZ_main") for b in boards) / len(boards)
            rec["star_ratio"] = boards.count("STAR") / len(boards)
            rec["chinext_ratio"] = boards.count("ChiNext") / len(boards)
            rec["board_hhi"] = sum((boards.count(b) / len(boards)) ** 2 for b in set(boards))
        if not score_df.empty and industry_col is not None and STOCK_COL in score_df.columns:
            sdf = score_df[(score_df["model_id"].astype(str) == str(model_id)) & (score_df[DATE_COL] == dt) & (score_df[STOCK_COL].astype(str).isin(targets))]
            if len(sdf) > 0:
                vc = sdf[industry_col].astype(str).value_counts(normalize=True)
                rec["industry_hhi"] = float((vc ** 2).sum())
                rec["top_industry"] = str(vc.index[0])
                rec["top_industry_ratio"] = float(vc.iloc[0])
            if cap_col is not None and cap_col in sdf.columns:
                caps = pd.to_numeric(sdf[cap_col], errors="coerce")
                rec["target_cap_mean"] = float(caps.mean()) if caps.notnull().any() else np.nan
                rec["target_cap_median"] = float(caps.median()) if caps.notnull().any() else np.nan
        rows.append(rec)
    return pd.DataFrame(rows).sort_values(DATE_COL)

exposure_drift_df = build_exposure_drift(monthly_all, score_panel_df, PRIMARY_MODEL_ID)
display(exposure_drift_df.tail(12))

## 5. 健康状态：Green / Yellow / Red

设计原则：

- 简洁，不做复杂黑箱评分。
- 不把单月噪声当结论，默认看 6 个月滚动。
- 使用历史扩展分位数判断 `Recall@30` 是否已经掉到自己的低位。

初版规则：

- **Green**：6 个月随机分位均值 >= 0.55，滚动 RankIC >= 0，且 Recall@30 不低于过去中位数。
- **Red**：6 个月随机分位均值 < 0.45，或 RankIC < -0.02，或 Recall@30 跌到过去 20% 分位以下。
- **Yellow**：其他情况。

这些阈值不是策略参数，必须通过下一节的历史回放验证；如果 Green/Red 不能区分未来表现，就不能用来做仓位动作。

In [ ]:
def add_trailing_health_state(df: pd.DataFrame, lookback: int = 6) -> pd.DataFrame:
    out = df.copy().sort_values(DATE_COL).reset_index(drop=True)
    metrics = ["random_percentile", "rank_ic", "recall30_real_top10", "selected_hit_rate_real_top20"]
    for c in metrics:
        if c in out.columns:
            out["trail_{}".format(c)] = pd.to_numeric(out[c], errors="coerce").rolling(lookback, min_periods=max(3, lookback//2)).mean()
    states = []
    reasons = []
    for i, row in out.iterrows():
        hist = out.iloc[:i]
        recall_hist = pd.to_numeric(hist.get("recall30_real_top10", pd.Series(dtype=float)), errors="coerce").dropna()
        hit_hist = pd.to_numeric(hist.get("selected_hit_rate_real_top20", pd.Series(dtype=float)), errors="coerce").dropna()
        recall_med = recall_hist.median() if len(recall_hist) >= 6 else 0.10
        recall_q20 = recall_hist.quantile(0.20) if len(recall_hist) >= 6 else 0.03
        hit_med = hit_hist.median() if len(hit_hist) >= 6 else 0.05
        rp = row.get("trail_random_percentile", np.nan)
        ric = row.get("trail_rank_ic", np.nan)
        rec = row.get("trail_recall30_real_top10", np.nan)
        hit = row.get("trail_selected_hit_rate_real_top20", np.nan)
        red_flags = []
        green_flags = []
        available_votes = 0
        if pd.notnull(rp):
            available_votes += 1
            if rp < 0.45:
                red_flags.append("random_percentile_weak")
            elif rp >= 0.55:
                green_flags.append("random_percentile_good")
        if pd.notnull(ric):
            available_votes += 1
            if ric < -0.02:
                red_flags.append("rank_ic_negative")
            elif ric >= 0.03:
                green_flags.append("rank_ic_good")
        if pd.notnull(rec):
            available_votes += 1
            if rec < recall_q20:
                red_flags.append("recall_low_vs_history")
            elif rec >= recall_med:
                green_flags.append("recall_ok_vs_history")
        if pd.notnull(hit):
            available_votes += 1
            if hit <= 0 and i >= lookback:
                red_flags.append("selected_hit_zero")
            elif hit >= hit_med and hit > 0:
                green_flags.append("selected_hit_ok")
        if available_votes < 2:
            states.append("Yellow")
            reasons.append("insufficient_history")
        elif red_flags and len(red_flags) >= max(1, available_votes // 2):
            states.append("Red")
            reasons.append(";".join(red_flags))
        elif len(green_flags) >= max(2, int(np.ceil(available_votes * 0.6))):
            states.append("Green")
            reasons.append(";".join(green_flags))
        else:
            states.append("Yellow")
            reasons.append("insufficient_or_mixed")
    out["health_state"] = states
    out["health_reason"] = reasons
    return out

health_state_df = add_trailing_health_state(health_monthly_df, LOOKBACK_MONTHS)
display(health_state_df.tail(18))
print(health_state_df["health_state"].value_counts(dropna=False))

## 6. 验证健康监测是否靠谱

这一步是 V62 的核心。

我们不是直接相信 Green/Yellow/Red，而是做历史回放：

- 每个月根据过去已完成月份计算健康状态。
- 然后看该状态之后未来 1/3/6 个月的策略表现。
- 如果 Red 之后没有更差，Green 之后没有更好，健康状态只能作为解释工具，不能作为仓位/持仓数控制信号。

In [ ]:
def forward_compound(values: pd.Series, start_pos: int, horizon: int) -> float:
    seg = pd.to_numeric(values.iloc[start_pos + 1:start_pos + 1 + horizon], errors="coerce").dropna()
    if len(seg) == 0:
        return np.nan
    return float((1 + seg).prod() - 1)


def add_forward_returns(df: pd.DataFrame, horizons: List[int]) -> pd.DataFrame:
    out = df.copy().sort_values(DATE_COL).reset_index(drop=True)
    for h in horizons:
        out[f"fwd_{h}m_raw_ret"] = [forward_compound(out["audit_raw_ret"], i, h) for i in range(len(out))]
        out[f"fwd_{h}m_excess_ret"] = [forward_compound(out["audit_excess_ret"], i, h) for i in range(len(out))]
    return out

health_validation_panel_df = add_forward_returns(health_state_df, FORWARD_HORIZONS)

summary_rows = []
for state, g in health_validation_panel_df.groupby("health_state"):
    rec = {"health_state": state, "months": int(len(g))}
    for h in FORWARD_HORIZONS:
        for kind in ["raw", "excess"]:
            col = f"fwd_{h}m_{kind}_ret"
            vals = pd.to_numeric(g[col], errors="coerce").dropna()
            rec[f"{col}_mean"] = float(vals.mean()) if len(vals) else np.nan
            rec[f"{col}_median"] = float(vals.median()) if len(vals) else np.nan
            rec[f"{col}_win_rate"] = float((vals > 0).mean()) if len(vals) else np.nan
            rec[f"{col}_count"] = int(len(vals))
    summary_rows.append(rec)
health_validation_summary_df = pd.DataFrame(summary_rows).sort_values("health_state")
display(health_validation_summary_df)

In [ ]:
def random_state_validation(df: pd.DataFrame, n_iter: int = 500, seed: int = 42) -> pd.DataFrame:
    rng = np.random.RandomState(seed)
    out = []
    state_order = ["Green", "Yellow", "Red"]
    present = [s for s in state_order if s in set(df["health_state"].dropna())]
    if len(present) < 2:
        return pd.DataFrame()
    good_state = "Green" if "Green" in present else present[0]
    bad_state = "Red" if "Red" in present else present[-1]
    for h in FORWARD_HORIZONS:
        col = f"fwd_{h}m_excess_ret"
        base = df[["health_state", col]].dropna()
        base = base[base["health_state"].isin([good_state, bad_state])].copy()
        if base.empty or base["health_state"].nunique() < 2:
            continue
        actual = base.groupby("health_state")[col].mean().to_dict()
        actual_spread = actual.get(good_state, np.nan) - actual.get(bad_state, np.nan)
        spreads = []
        for _ in range(n_iter):
            shuffled = base.copy()
            shuffled["health_state"] = rng.permutation(shuffled["health_state"].values)
            means = shuffled.groupby("health_state")[col].mean().to_dict()
            spreads.append(means.get(good_state, np.nan) - means.get(bad_state, np.nan))
        spreads = pd.Series(spreads).dropna()
        p_value = float((spreads >= actual_spread).mean()) if pd.notnull(actual_spread) and len(spreads) else np.nan
        out.append({
            "horizon": h,
            "good_state": good_state,
            "bad_state": bad_state,
            "actual_good_minus_bad_excess": actual_spread,
            "random_spread_mean": float(spreads.mean()) if len(spreads) else np.nan,
            "random_spread_std": float(spreads.std()) if len(spreads) else np.nan,
            "one_sided_p_good_better_than_bad": p_value,
            "n_random": int(len(spreads)),
        })
    return pd.DataFrame(out)

health_random_validation_df = random_state_validation(health_validation_panel_df, RANDOM_N, RANDOM_SEED)
display(health_random_validation_df)

## 7. 输出文件

重点看：

- `v62_model_alpha_audit.csv`：各模型统一收益审计。
- `v62_monthly_health_metrics.csv`：每月健康指标。
- `v62_health_state.csv`：滚动健康状态。
- `v62_health_validation_summary.csv`：Green/Yellow/Red 对未来收益是否有区分度。
- `v62_health_random_validation.csv`：健康状态有效性的随机置换检验。
- `v62_exposure_drift.csv`：目标组合暴露漂移。

In [ ]:
model_alpha_audit_df.to_csv(OUT_DIR / "v62_model_alpha_audit.csv", index=False)
health_monthly_df.to_csv(OUT_DIR / "v62_monthly_health_metrics.csv", index=False)
health_state_df.to_csv(OUT_DIR / "v62_health_state.csv", index=False)
health_validation_panel_df.to_csv(OUT_DIR / "v62_health_validation_panel.csv", index=False)
health_validation_summary_df.to_csv(OUT_DIR / "v62_health_validation_summary.csv", index=False)
health_random_validation_df.to_csv(OUT_DIR / "v62_health_random_validation.csv", index=False)
exposure_drift_df.to_csv(OUT_DIR / "v62_exposure_drift.csv", index=False)

print("saved files:")
for p in sorted(OUT_DIR.glob("v62_*.csv")):
    print("-", p)

## 8. 读数规则

先按这个顺序读结果：

1. `v62_model_alpha_audit.csv`：确认主模型在 JQ-like/realized/proxy 口径下的累计收益、超额、回撤和 drop-top stress。
2. `v62_monthly_health_metrics.csv`：看 `Recall@30_realTop10`、`selected_hit_real_top20`、`random_percentile`、`rank_ic` 是否同向。
3. `v62_health_validation_summary.csv`：如果 Green 后未来收益明显好于 Red，健康监测才有资格进入 V64 的组合规则。
4. `v62_health_random_validation.csv`：如果 Green-Red 差异没有通过随机置换检验，健康状态只能作为解释和预警，不能作为仓位控制。
5. `v62_exposure_drift.csv`：如果收益好时总是对应市值/行业/板块极端漂移，需要把它视为风格暴露，不直接归因为模型 alpha。

进入下一步的条件：

- 如果 V62 显示健康指标有预测价值：V64 可以测试健康状态驱动的 topN/分散度规则。
- 如果健康指标没有预测价值：V64 只做静态组合层优化，不使用 Green/Yellow/Red 控制仓位。
- 如果主模型相对随机组合长期没有优势：优先回到模型/特征主线，而不是继续组合微调。